# Project: Image Style Transfer

## 1. Problem Statement

The goal of this project is to create a system that applies the artistic style of Vincent van Gogh to ordinary photographs. The model should preserve the main content and structure of the original image while transforming its visual appearance to match Van Gogh’s painting style. Deep learning techniques are used to learn and transfer characteristic patterns such as colors, brush strokes, and textures. The final system should generate visually appealing stylized images in an efficient and automatic way.

## 2. Import Libraries and Define Constants

In [41]:
import matplotlib.pyplot as plt
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms, models
from tqdm import tqdm
from PIL import Image

is_debug_mode = False
print('Mode:', 'debug' if is_debug_mode else 'release')

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

img_size = 256 if is_debug_mode else 640
print('Image size: ', img_size)

num_epochs = 10 if is_debug_mode else 20
print('Num epochs:', num_epochs)

batch_size = 4

Mode: release
Using device: mps
Image size:  640
Num epochs: 20


In [42]:
class StyleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 9, padding=4),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),      # downsample
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),     # downsample
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(inplace=True),

            # Upsample back to original size
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(32, 3, 9, padding=4),
            nn.Tanh()  # output in [-1, 1]
        )

    def forward(self, x):
        return self.net(x)

In [43]:
class VGGFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features
        self.layers = nn.ModuleList(vgg[:29])
        self.targets = [0, 5, 10, 19, 28]

        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        feats = []
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i in self.targets:
                feats.append(x)
        return feats


In [44]:
def gram_matrix(f):
    B, C, H, W = f.size()
    f = f.view(B, C, H * W)
    g = torch.bmm(f, f.transpose(1, 2))
    return g / (C * H * W)


In [45]:
transform = transforms.Compose([
    transforms.Resize(img_size+20),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

content_dataset = ImageFolder("data/content/", transform=transform)
content_loader = DataLoader(content_dataset, batch_size=4, shuffle=True)

style_image = Image.open("data/style.jpg").convert("RGB")
style_image = transform(style_image).unsqueeze(0).to(device)

In [46]:
style_net = StyleNet().to(device)
vgg = VGGFeatures().to(device).eval()

optimizer = optim.Adam(style_net.parameters(), lr=1e-3)
mse = nn.MSELoss()

In [47]:
with torch.no_grad():
    style_features = vgg(style_image)
    style_grams = [gram_matrix(f) for f in style_features]


In [ ]:
STYLE_WEIGHT = 1e6
CONTENT_WEIGHT = 1.0

for epoch in range(num_epochs):
    for content_imgs, _ in content_loader:
        content_imgs = content_imgs.to(device)

        stylized = style_net(content_imgs)

        content_feats = vgg(content_imgs)
        stylized_feats = vgg(stylized)

        # Content loss (relu3_1)
        content_loss = mse(
            stylized_feats[2],
            content_feats[2]
        )

        # Style loss
        style_loss = 0.0
        for sf, sg in zip(stylized_feats, style_grams):
            G = gram_matrix(sf)
            style_loss += mse(G, sg.expand_as(G))

        loss = CONTENT_WEIGHT * content_loss + STYLE_WEIGHT * style_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} | Content {content_loss.item():.3f} | Style {style_loss.item():.3f}")


In [ ]:
torch.save(style_net.state_dict(), "vangogh_style_net.pth")


In [ ]:
model = StyleNet().to(device)
model.load_state_dict(torch.load("vangogh_style_net.pth"))
model.eval()

img = Image.open("data/mountains.jpeg").convert("RGB")
img = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    out = model(img)

def denorm(x):
    mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(device)
    std  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(device)
    return (x * std + mean).clamp(0,1)

out_img = denorm(out)[0].cpu()
transforms.ToPILImage()(out_img).save("data/van_gogh_result.png")
